In [8]:
import gurobipy as gurobi
from gurobipy import GRB

### Model Formulation

Decision Variables

$$x_{t,r} \in \{0,1\} \quad \forall\, t \in T,\; r \in R$$

Objective: maximize total NPV

$$\max \sum_{t \in T}\sum_{r \in R} \text{NPV}_{t,r}\, x_{t,r}$$

Constraints

$$\sum_{t,r} \text{Cap}_{t,r}\, x_{t,r} \le 15 \quad \text{(budget)}$$

$$\sum_{t,r} x_{t,r} \le 2 \quad \text{(no more than two projects)}$$

$$\sum_{t \in T} x_{t,r} \le 1 \quad \forall\, r \in R \quad \text{(one per region)}$$

In [5]:
projects = [
    ("Solar Farm","North",4,5),   ("Solar Farm","Central",3.5,4), ("Solar Farm","South",4.5,6),
    ("Wind Turbine","North",5.5,7),("Wind Turbine","Central",4,5), ("Wind Turbine","South",5,6),
    ("Geothermal","North",6,8),    ("Geothermal","Central",4.5,6), ("Geothermal","South",6.5,9),
]

types = ["Solar Farm", "Wind Turbine", "Geothermal"]
regions = ["North", "Central", "South"]

m = gurobi.Model("EcoPower")

x = {(t, r): m.addVar(vtype = GRB.BINARY, name = f"x_{t}_{r}".replace(" ", "_")) for (t, r, npv, cap) in projects}

m.setObjective(gurobi.quicksum(npv * x[t, r] for (t, r, npv, cap) in projects), GRB.MAXIMIZE)

m.addConstr(gurobi.quicksum(cap*x[(t,r)] for (t,r,npv,cap) in projects) <= 15, "budget")
m.addConstr(gurobi.quicksum(x[(t,r)] for (t,r,npv,cap) in projects) <= 2, "max_projects")
for r in regions:
    m.addConstr(gurobi.quicksum(x[(t, r)] for t in types) <= 1, f"one_project_per_region_{r}")

m.optimize()
print(f"\nOptimal total NPV: ${m.ObjVal} million\n")
print("Selected projects:")
for (t,r,npv,cap) in projects:
    if x[(t,r)].X > 0.5:
        print(f"  {t} in {r}  (NPV=${npv}M, Capital=${cap}M)")

Set parameter Username
Set parameter LicenseID to value 2859698
Academic license - for non-commercial use only - expires 2027-09-02
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 5 rows, 9 columns and 27 nonzeros (Max)
Model fingerprint: 0x4db3d5a3
Model has 9 linear objective coefficients
Variable types: 0 continuous, 9 integer (9 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [4e+00, 7e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Found heuristic solution: objective 7.5000000
Presolve removed 0 rows and 2 columns
Presolve time: 0.03s
Presolved: 5 rows, 7 columns, 21 nonzeros
Variable types: 0 continuous, 7 integer (7 binary)

Root relaxation: objective 1.150000e+01, 3 iterations, 0.01 seconds (0.0

Sets:

S = {1,...,7}: the seven three-day shift patterns
D = {Mon,...,Sun}: days of the week

Parameters:

$w_s$: daily wage per worker on shift $s$
$c = 800$: part-time rate per day
$r_d$: crew members required on day $d$
$a_{s,d} = 1$ if shift $s$ works on day $d$, else $0$

Decision variables:

$x_s \ge 0$, integer, for all $s \in S$ (workers assigned to shift $s$)
$p_d \ge 0$, integer, for all $d \in D$ (part-time workers hired on day $d$)

Objective (minimize total cost; a shift worker is paid their daily wage on all three days, so shift $s$ costs $3 w_s$ per worker):

$$\min \sum_{s \in S} 3 w_s x_s + \sum_{d \in D} c\, p_d$$

Constraints (staffing requirement met each day):

$$\sum_{s \in S} a_{s,d} x_s + p_d \ge r_d \quad \text{for all } d \in D$$

Nonnegativity and integrality:

$$x_s \ge 0, \quad p_d \ge 0, \quad x_s, p_d

In [7]:
days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
req = {"Mon":150,"Tue":180,"Wed":200,"Thu":220,"Fri":250,"Sat":300,"Sun":100}

shifts = {
    1:(["Mon","Tue","Wed"], 600),
    2:(["Tue","Wed","Thu"], 550),
    3:(["Wed","Thu","Fri"], 650),
    4:(["Thu","Fri","Sat"], 620),
    5:(["Fri","Sat","Sun"], 580),
    6:(["Mon","Thu","Sat"], 700),
    7:(["Tue","Fri","Sun"], 680),
}

m = gurobi.Model("airport_staffing")

x = {s: m.addVar(vtype=GRB.INTEGER, lb=0, name=f"x{s}") for s in shifts}
p = {d: m.addVar(vtype=GRB.INTEGER, lb=0, name=f"p_{d}") for d in days}

m.setObjective(
    gurobi.quicksum(3 * wage * x[s] for s, (dw, wage) in shifts.items())
    + gurobi.quicksum(800 * p[d] for d in days),
    GRB.MINIMIZE
)

for d in days:
    m.addConstr(
        gurobi.quicksum(x[s] for s, (dw, wage) in shifts.items() if d in dw) + p[d] >= req[d],
        name=f"req_{d}"
    )

m.optimize()

print(f"\nOptimal total cost: ${m.ObjVal:,.0f}\n")
print("Shifts staffed:")
for s in shifts:
    if x[s].X > 0.5:
        print(f"  Shift {s}: {int(round(x[s].X))} workers")
print("\nPart-time hires:")
any_pt = False
for d in days:
    if p[d].X > 0.5:
        print(f"  {d}: {int(round(p[d].X))} workers")
        any_pt = True
if not any_pt:
    print("  None")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 7 rows, 14 columns and 28 nonzeros (Min)
Model fingerprint: 0x143ca96b
Model has 14 linear objective coefficients
Variable types: 0 continuous, 14 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [8e+02, 2e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 3e+02]

Found heuristic solution: objective 1120000.0000
Presolve time: 0.00s
Presolved: 7 rows, 14 columns, 28 nonzeros
Variable types: 0 continuous, 14 integer (0 binary)

Root relaxation: objective 8.515000e+05, 8 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Ti

Sets:

I = {1,...,15}: items to pack
B = {1,...,8}: available boxes

Parameters:

$v_i$: volume of item $i$
$C = 50$: capacity of each box

Decision variables:

$x_{i,b} = 1$ if item $i$ is placed in box $b$, else $0$
$y_b = 1$ if box $b$ is used, else $0$

Objective (minimize boxes used):

$$\min \sum_{b \in B} y_b$$

Constraints:

Each item is assigned to exactly one box:

$$\sum_{b \in B} x_{i,b} = 1 \quad \text{for all } i \in I$$

Box capacity is not exceeded, and a box holding any item must be counted as used:

$$\sum_{i \in I} v_i x_{i,b} \le C y_b \quad \text{for all } b \in B$$

Binary restrictions:

$$x_{i,b} \in \{0,1\}, \quad y_b \in \{0,1\}$$

In [ ]:
vol = {1:12,2:7,3:19,4:10,5:24,6:8,7:15,8:22,9:9,10:14,11:11,12:18,13:16,14:13,15:17}
items = list(vol)
boxes = list(range(1,9))
C = 50

m = gurobi.Model("moving_boxes")

x = {(i,b): m.addVar(vtype=GRB.BINARY, name=f"x_{i}_{b}") for i in items for b in boxes}
y = {b: m.addVar(vtype=GRB.BINARY, name=f"y_{b}") for b in boxes}

m.setObjective(gurobi.quicksum(y[b] for b in boxes), GRB.MINIMIZE)

for i in items:
    m.addConstr(gurobi.quicksum(x[(i,b)] for b in boxes) == 1, name=f"assign_{i}")
for b in boxes:
    m.addConstr(gurobi.quicksum(vol[i]*x[(i,b)] for i in items) <= C*y[b], name=f"cap_{b}")

m.optimize()

print(f"\nMinimum boxes used: {int(m.ObjVal)}\n")
for b in boxes:
    if y[b].X > 0.5:
        packed = [i for i in items if x[(i,b)].X > 0.5]
        print(f"  Box {b}: items {packed}, total volume {sum(vol[i] for i in packed)}")

Sets:

S = {1,...,8}: security shifts
K = {1,...,8}: available guard labels (at most one per shift)
E: set of conflicting shift pairs $(a,b)$

Decision variables:

$x_{s,k} = 1$ if shift $s$ is assigned to guard $k$, else $0$
$w_k = 1$ if guard $k$ is used, else $0$

Objective (minimize guards hired):

$$\min \sum_{k \in K} w_k$$

Constraints:

Each shift is covered by exactly one guard:

$$\sum_{k \in K} x_{s,k} = 1 \quad \text{for all } s \in S$$

Two conflicting shifts cannot share a guard, and any guard covering a shift must be counted as used:

$$x_{a,k} + x_{b,k} \le w_k \quad \text{for all } (a,b) \in E,\ k \in K$$

Binary restrictions:

$$x_{s,k} \in \{0,1\}, \quad w_k \in \{0,1\}$$

In [ ]:
conflicts = {1:[2,3], 2:[1,3,4], 3:[1,2,5], 4:[2,5,6], 5:[3,4,6,7], 6:[4,5,7], 7:[5,6,8], 8:[7]}
shifts = list(conflicts)
guards = list(range(1,9))

edges = set()
for s, cs in conflicts.items():
    for c in cs:
        edges.add((min(s,c), max(s,c)))

m = gurobi.Model("festival_security")

x = {(s,k): m.addVar(vtype=GRB.BINARY, name=f"x_{s}_{k}") for s in shifts for k in guards}
w = {k: m.addVar(vtype=GRB.BINARY, name=f"w_{k}") for k in guards}

m.setObjective(gurobi.quicksum(w[k] for k in guards), GRB.MINIMIZE)

for s in shifts:
    m.addConstr(gurobi.quicksum(x[(s,k)] for k in guards) == 1, name=f"cover_{s}")
for (a,b) in edges:
    for k in guards:
        m.addConstr(x[(a,k)] + x[(b,k)] <= w[k], name=f"conflict_{a}_{b}_{k}")

m.optimize()

print(f"\nMinimum guards needed: {int(m.ObjVal)}\n")
for k in guards:
    if w[k].X > 0.5:
        assigned = [s for s in shifts if x[(s,k)].X > 0.5]
        print(f"  Guard {k}: shifts {assigned}")